# MindForge Taxonomy Refresh (Bilingual)

This notebook builds and refreshes a stable taxonomy for markdown exports in `intermediate_markdowns/`.

The taxonomy refresh uses only frontmatter fields from each file:
- `original_title`
- `generated_title`
- `summary`

Goals:
- Keep categories stable across reruns
- Detect when new conversations require new categories
- Persist only the taxonomy as the source of truth
- Keep labels consistent in one language (Chinese-first by default)

## Model Recommendation (Free First)

Primary free choice for Chinese-first + solid English:
- `Ollama + qwen2.5:7b-instruct` (local, no API token cost)

Alternative cloud route:
- Any OpenAI-compatible endpoint with a free-tier Qwen model

Why this works:
- Qwen is strong in Simplified Chinese and good enough in English for mixed corpora
- Constrained JSON + category enums keeps output consistent

In [8]:
from __future__ import annotations

import json
import os
import re
import hashlib
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any
from collections import Counter

import requests

In [2]:
# --- Configuration ---
PROJECT_ROOT = Path.cwd()
INTERMEDIATE_DIR = PROJECT_ROOT / "intermediate_markdowns"
STATE_DIR = PROJECT_ROOT / "taxonomy_state"
STATE_DIR.mkdir(exist_ok=True)

TAXONOMY_FILE = STATE_DIR / "taxonomy_v1.json"

# Provider: 'ollama' (free local) or 'openai_compatible'
PROVIDER = os.getenv("TAXONOMY_PROVIDER", "ollama")

# Ollama settings (recommended default)
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")

# OpenAI-compatible settings (optional)
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "")

# Taxonomy behavior
TARGET_LABEL_LANGUAGE = "zh"  # Keep category labels in Chinese for consistency
MAX_DISCOVERY_DOCS = 60

print({
    "project_root": str(PROJECT_ROOT),
    "corpus_dir": str(INTERMEDIATE_DIR),
    "provider": PROVIDER,
    "model": OLLAMA_MODEL if PROVIDER == "ollama" else OPENAI_MODEL,
})

{'project_root': 'c:\\Users\\HELEN1822\\OneDrive - Willis Towers Watson\\Documents\\Github\\MindForge', 'corpus_dir': 'c:\\Users\\HELEN1822\\OneDrive - Willis Towers Watson\\Documents\\Github\\MindForge\\intermediate_markdowns', 'provider': 'ollama', 'model': 'qwen2.5:7b-instruct'}


In [3]:
# --- Utilities ---
def extract_frontmatter_and_body(text: str) -> tuple[dict[str, Any], str]:
    # Normalize newlines so frontmatter parsing is stable across platforms.
    normalized = text.replace("\r\n", "\n")
    if normalized.startswith("---\n"):
        parts = normalized.split("---\n", 2)
        if len(parts) == 3:
            fm_raw, body = parts[1], parts[2]
            fm = {}
            for line in fm_raw.splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    fm[k.strip()] = v.strip().strip('"')
            return fm, body
    return {}, normalized

def detect_language_simple(text: str) -> str:
    zh_chars = len(re.findall(r"[\u4e00-\u9fff]", text))
    en_chars = len(re.findall(r"[A-Za-z]", text))
    total = zh_chars + en_chars
    if total == 0:
        return "unknown"
    zh_ratio = zh_chars / total
    if zh_ratio > 0.65:
        return "zh"
    if zh_ratio < 0.35:
        return "en"
    return "mixed"

def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:16]

def load_markdown_corpus(corpus_dir: Path) -> list[dict[str, Any]]:
    docs = []
    for p in sorted(corpus_dir.glob("*.md")):
        raw = p.read_text(encoding="utf-8", errors="ignore")
        frontmatter, _ = extract_frontmatter_and_body(raw)

        original_title = frontmatter.get("original_title", p.stem).strip()
        generated_title = frontmatter.get("generated_title", original_title).strip()
        summary = frontmatter.get("summary", "").strip()

        language = detect_language_simple(" ".join([original_title, generated_title, summary]))
        signature = json.dumps(
            {
                "original_title": original_title,
                "generated_title": generated_title,
                "summary": summary,
            },
            ensure_ascii=False,
            sort_keys=True,
        )

        docs.append({
            "path": str(p.relative_to(PROJECT_ROOT)),
            "filename": p.name,
            "original_title": original_title,
            "generated_title": generated_title,
            "summary": summary,
            "language": language,
            "content_hash": stable_hash(signature),
        })
    return docs

In [5]:
# --- LLM clients ---
def provider_preflight_check(raise_on_fail: bool = True) -> dict[str, Any]:
    result = {"provider": PROVIDER, "ok": True, "message": ""}

    if PROVIDER == "ollama":
        try:
            r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=8)
            r.raise_for_status()
            models = [m.get("name", "") for m in r.json().get("models", [])]
            has_model = any(name == OLLAMA_MODEL or name.startswith(f"{OLLAMA_MODEL}:") for name in models)
            if not has_model:
                result["ok"] = False
                result["message"] = (
                    f"Ollama is reachable but model '{OLLAMA_MODEL}' was not found. "
                    f"Available models: {models[:8]}"
                )
            else:
                result["message"] = f"Ollama ready at {OLLAMA_BASE_URL} with model {OLLAMA_MODEL}."
        except requests.RequestException as e:
            result["ok"] = False
            result["message"] = (
                f"Cannot reach Ollama at {OLLAMA_BASE_URL}. Start Ollama and ensure port is open. "
                f"Original error: {e}"
            )

    elif PROVIDER == "openai_compatible":
        missing = [k for k, v in {
            "OPENAI_BASE_URL": OPENAI_BASE_URL,
            "OPENAI_API_KEY": OPENAI_API_KEY,
            "OPENAI_MODEL": OPENAI_MODEL,
        }.items() if not v]
        if missing:
            result["ok"] = False
            result["message"] = f"Missing required settings for openai_compatible: {missing}"
        else:
            try:
                url = OPENAI_BASE_URL.rstrip("/") + "/models"
                headers = {"Authorization": f"Bearer {OPENAI_API_KEY}"}
                r = requests.get(url, headers=headers, timeout=8)
                r.raise_for_status()
                result["message"] = f"OpenAI-compatible endpoint reachable: {OPENAI_BASE_URL}"
            except requests.RequestException as e:
                result["ok"] = False
                result["message"] = (
                    f"Cannot reach openai_compatible endpoint at {OPENAI_BASE_URL}. "
                    f"Original error: {e}"
                )

    else:
        result["ok"] = False
        result["message"] = f"Unsupported provider: {PROVIDER}"

    if raise_on_fail and not result["ok"]:
        raise RuntimeError(result["message"])
    return result

def llm_call_json(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> dict[str, Any]:
    if PROVIDER == "ollama":
        payload = {
            "model": OLLAMA_MODEL,
            "prompt": f"<|system|>\n{system_prompt}\n<|user|>\n{user_prompt}\n<|assistant|>",
            "stream": False,
            "options": {"temperature": temperature},
            "format": "json",
        }
        try:
            r = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=180)
            r.raise_for_status()
            text = r.json().get("response", "{}")
            return json.loads(text)
        except requests.RequestException as e:
            raise RuntimeError(
                f"Ollama request failed at {OLLAMA_BASE_URL} using model {OLLAMA_MODEL}. "
                "Run preflight, start Ollama, and confirm the model is installed. "
                f"Original error: {e}"
            ) from e
        except json.JSONDecodeError as e:
            preview = text[:300] if 'text' in locals() else "<empty>"
            raise RuntimeError(
                f"Ollama returned non-JSON output. Response preview: {preview}"
            ) from e

    if PROVIDER == "openai_compatible":
        if not OPENAI_BASE_URL or not OPENAI_API_KEY or not OPENAI_MODEL:
            raise ValueError("Set OPENAI_BASE_URL, OPENAI_API_KEY, OPENAI_MODEL first")

        url = OPENAI_BASE_URL.rstrip("/") + "/chat/completions"
        headers = {
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": OPENAI_MODEL,
            "temperature": temperature,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        }
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=180)
            r.raise_for_status()
            text = r.json()["choices"][0]["message"]["content"]
            return json.loads(text)
        except requests.RequestException as e:
            raise RuntimeError(
                f"openai_compatible request failed at {url} with model {OPENAI_MODEL}. "
                "Check endpoint, API key, and model name. "
                f"Original error: {e}"
            ) from e
        except (KeyError, IndexError, json.JSONDecodeError) as e:
            preview = text[:300] if 'text' in locals() else "<empty>"
            raise RuntimeError(
                f"openai_compatible returned unexpected non-JSON content. Response preview: {preview}"
            ) from e

    raise ValueError(f"Unsupported provider: {PROVIDER}")

In [9]:
# --- Stage A: Discover / Refresh taxonomy ---
def build_or_refresh_taxonomy(docs: list[dict[str, Any]]) -> dict[str, Any]:
    sampled = docs[:MAX_DISCOVERY_DOCS]
    language_stats = Counter(d["language"] for d in docs)

    existing = None
    if TAXONOMY_FILE.exists():
        existing = json.loads(TAXONOMY_FILE.read_text(encoding="utf-8"))

    system_prompt = (
        "You are a taxonomy designer for conversation notes. Return strict JSON only. "
        "Keep stable categories if existing taxonomy is provided. "
        "Prefer concise labels in Simplified Chinese when target language is zh."
    )

    user_prompt = json.dumps({
        "task": "Build or refresh taxonomy",
        "target_label_language": TARGET_LABEL_LANGUAGE,
        "rules": [
            "Preserve existing categories unless strongly necessary",
            "Add at most 3 new categories in one refresh",
            "Every category must include: category_id, label, description, include_examples, exclude_examples",
            "category_id must be stable snake_case ascii",
            "Include fallback categories: other, unclear",
            "Infer topics from only original_title, generated_title, and summary",
        ],
        "existing_taxonomy": existing,
        "language_stats": language_stats,
        "documents": [
            {
                "original_title": d["original_title"],
                "generated_title": d["generated_title"],
                "summary": d["summary"],
            }
            for d in sampled
        ],
        "output_schema": {
            "version": "string",
            "generated_at": "iso8601",
            "target_label_language": "zh|en",
            "categories": [
                {
                    "category_id": "snake_case",
                    "label": "string",
                    "description": "string",
                    "include_examples": ["string"],
                    "exclude_examples": ["string"],
                }
            ],
            "change_summary": {
                "kept": ["category_id"],
                "added": ["category_id"],
                "deprecated": ["category_id"],
            },
        },
    }, ensure_ascii=False)

    taxonomy = llm_call_json(system_prompt, user_prompt, temperature=0.0)
    taxonomy.setdefault("version", "v1")
    taxonomy["generated_at"] = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    taxonomy["language_stats"] = dict(language_stats)

    TAXONOMY_FILE.write_text(json.dumps(taxonomy, ensure_ascii=False, indent=2), encoding="utf-8")
    return taxonomy

In [8]:
# --- Optional test: classify one markdown note (print only, no save) ---
def tag_one_markdown(md_path: str | None = None) -> dict[str, Any]:
    if not TAXONOMY_FILE.exists():
        raise FileNotFoundError(f"Taxonomy file not found: {TAXONOMY_FILE}")

    taxonomy = json.loads(TAXONOMY_FILE.read_text(encoding="utf-8"))
    categories = taxonomy.get("categories", [])
    category_ids = [c["category_id"] for c in categories]
    if not category_ids:
        raise ValueError("Taxonomy has no categories")

    if md_path is None:
        candidates = sorted(EXPORT_DIR.glob("*.md"))
        if not candidates:
            raise FileNotFoundError(f"No markdown files found in {EXPORT_DIR}")
        picked = candidates[0]
    else:
        picked = (PROJECT_ROOT / md_path).resolve()
        if not picked.exists():
            raise FileNotFoundError(f"Markdown file not found: {picked}")

    raw = picked.read_text(encoding="utf-8", errors="ignore")
    frontmatter, body = extract_frontmatter_and_body(raw)
    doc = {
        "path": str(picked.relative_to(PROJECT_ROOT)),
        "title": frontmatter.get("title", picked.stem),
        "language": detect_language_simple(body),
        "snippet": clean_snippet(body),
    }

    system_prompt = (
        "You are a strict classifier. Return JSON only. "
        "Choose category_id from the provided enum only."
    )
    user_prompt = json.dumps({
        "task": "Classify one markdown note",
        "category_enum": category_ids,
        "categories": categories,
        "document": {
            "title": doc["title"],
            "language": doc["language"],
            "snippet": doc["snippet"],
        },
        "output_schema": {
            "primary_category_id": "enum",
            "secondary_category_id": "enum|null",
            "confidence": "0..1",
            "reason": "short sentence",
        },
    }, ensure_ascii=False)

    out = llm_call_json(system_prompt, user_prompt, temperature=0.0)

    primary = out.get("primary_category_id")
    if primary not in category_ids:
        primary = "unclear" if "unclear" in category_ids else category_ids[0]

    secondary = out.get("secondary_category_id")
    if secondary not in category_ids:
        secondary = None

    confidence = out.get("confidence", 0.0)
    if not isinstance(confidence, (int, float)):
        confidence = 0.0

    id_to_label = {c["category_id"]: c["label"] for c in categories}
    result = {
        "path": doc["path"],
        "title": doc["title"],
        "language": doc["language"],
        "primary_category_id": primary,
        "primary_label": id_to_label.get(primary, primary),
        "secondary_category_id": secondary,
        "secondary_label": id_to_label.get(secondary, secondary) if secondary else None,
        "confidence": round(float(confidence), 4),
        "reason": str(out.get("reason", ""))[:240],
    }
    return result

In [17]:
# --- Provider preflight check (run before refresh) ---
preflight = provider_preflight_check(raise_on_fail=False)
print(json.dumps(preflight, ensure_ascii=False, indent=2))
if not preflight.get("ok", False):
    raise RuntimeError("Provider preflight failed. Fix provider setup, then rerun this cell.")

{
  "provider": "ollama",
  "ok": true,
  "message": "Ollama ready at http://localhost:11434 with model qwen2.5:7b-instruct."
}


In [10]:
# --- Run taxonomy refresh only ---
docs = load_markdown_corpus(INTERMEDIATE_DIR)
print(f"Loaded {len(docs)} markdown files from {INTERMEDIATE_DIR}")
print("Language distribution:", dict(Counter(d['language'] for d in docs)))

provider_preflight_check()

taxonomy = build_or_refresh_taxonomy(docs)
print(f"Saved taxonomy to: {TAXONOMY_FILE}")
print(f"Category count: {len(taxonomy.get('categories', []))}")

Loaded 42 markdown files from c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\intermediate_markdowns
Language distribution: {'zh': 38, 'mixed': 3, 'en': 1}
Saved taxonomy to: c:\Users\HELEN1822\OneDrive - Willis Towers Watson\Documents\Github\MindForge\taxonomy_state\taxonomy_v1.json
Category count: 5


## Recurring Refresh Workflow

1. Add or update markdown files in `intermediate_markdowns/`
2. Re-run the taxonomy refresh cells
3. Compare `taxonomy_state/taxonomy_v1.json` change summary
4. If `other` grows too much, allow at most 1-3 new categories in next refresh

This notebook persists only taxonomy state and does not save classification results.

## One-File Tagging Check (No Save)

Use the final code cell to test one markdown note against the current taxonomy. The result is printed only for inspection and is not written to disk.

In [19]:
# --- Single-note tagging test (print only, no save) ---
# Leave as None to auto-pick the first markdown file in DeepSeek_Exports.
TEST_MD_PATH = None
# Example: TEST_MD_PATH = "DeepSeek_Exports/2026-04-20_xxxxxxxx.md"

can_run = True

if "tag_one_markdown" not in globals():
    can_run = False
    print("Tagging helper is not loaded. Run the helper cell above first.")

if not TAXONOMY_FILE.exists():
    can_run = False
    print(f"Taxonomy file missing: {TAXONOMY_FILE}")
    print("Run preflight and refresh first, then rerun this test cell.")

if can_run:
    try:
        test_result = tag_one_markdown(TEST_MD_PATH)
        print(json.dumps(test_result, ensure_ascii=False, indent=2))
    except Exception as e:
        print(f"Tagging test could not run: {e}")
        print("Next steps: run preflight, run refresh, then rerun this test cell.")

{
  "path": "DeepSeek_Exports\\2026-03-08 制定个性化护肤方案需求清单.md",
  "title": "2026-03-08 制定个性化护肤方案需求清单",
  "language": "zh",
  "primary_category_id": "情绪与心理问题",
  "primary_label": "情绪与心理问题",
  "secondary_category_id": null,
  "secondary_label": null,
  "confidence": 0.7,
  "reason": "虽然文档主要讨论护肤方案，但未涉及具体的情绪或心理问题，因此归类为情绪与心理问题类别较为合适。"
}
